In [1]:
import torch 
import numpy as np 
from pathlib import Path
import importlib
import IPython.display as ipd
import src.spatial_attn_lightning as binaural_lightning 
import yaml
import os
from pytorch_lightning import LightningModule, Trainer
import importlib

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:
from corpus.binaural_attention_h5 import BinauralAttentionDataset
import src.custom_modules as cm
import src.location_classifier_lightning as loc_lightning

importlib.reload(loc_lightning)

LocationClassifier = loc_lightning.LocationClassifier

In [3]:
config_path = "config/binaural_attn/word_task_standard_v07.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['model']['num_classes']['num_locs'] = 504 # 360 azimuth and 90 in elevation
del config['model']['num_classes']['num_words']
config['hparas']['batch_size'] = 32

config['corpus']['task'] = "location"
config['corpus']['skip_negative_elev'] = True
config['hparas']['lr'] = 0.001
config['num_workers'] = 2
ckpt_path = Path('attn_cue_models/word_task_standard_v07/checkpoints/epoch=3-step=67111.ckpt')

In [4]:
model = LocationClassifier(config, ckpt_path=ckpt_path)

center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


In [5]:
trainset = model.train_dataloader()

Using v06 dataset
/om/scratch/Sun/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v07
cue type: voice_and_location
1239 files in train concat dataset
len training set = 154875


In [6]:
trainer = Trainer(
    precision="32",
    limit_val_batches=0.0,
    num_nodes=1,
    # benchmark=True,
    devices=1, # was gpus=1,
    # detect_anomaly=True,
    # strategy="ddp_notebook",
    gradient_clip_val=100,
    accelerator="gpu",
)
trainer.fit(model)

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /om2/user/imgriff/conda_envs/pytorch_2/lib/python3.1 ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                         | Params
------------------------------------------------------------------
0 | audio_transforms | AudioCompose                 | 0     
1 | model            | OptimizedModule      

Using v06 dataset
/om/scratch/Sun/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v07
cue type: voice_and_location
1239 files in train concat dataset
len training set = 154875
Epoch 0:   0%|          | 411/154875 [05:41<35:36:10,  1.21it/s, v_num=3.52e+7, train_loss=3.04e+3, grad_norm=2.33e+4]

Process ForkProcess-1:
Process ForkProcess-2:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/concurrent/futures/process.py", line 244, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/concurrent/futures/process.py", line 244, in 

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/trainer/call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...
